# 08 - Recover the SCiO scan-encryption key (firmware key recovery)

The raw scan blobs are DSP-binned and then **encrypted on the device**. No
version of the Consumer Physics app decrypts them, and the server is gone, so the
only offline path to a spectrum is to recover the device's encryption key from
its **DSP firmware** (the Blackfin BF512 `dsp_op` file).

This notebook:
1. loads firmware/table blobs you extracted from an old phone (or an `adb` backup);
2. triages them and scans for cipher signatures;
3. runs `scio.keyrecover` to test firmware-derived candidate keys against your
   captured scans, using a plaintext oracle (a correct key turns the ciphered
   body into a smooth binned intensity vector);
4. if the key is found, decrypts a scan and compares a fixture's derived
   reflectance to the server's stored spectrum.

No key spaces are brute-forced: only constants actually present in the firmware
(or standard derivations of device identifiers) are tried.

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))
from scio import firmware, keyrecover, store, decode

## 1. Get the firmware blobs onto disk

The DSP firmware and per-device tables were cached by the SCiO apps in Android
SharedPreferences. On a phone that ran the app (consumer or Lab), the files live
under `/data/data/com.consumerphysics.*/shared_prefs/*.xml` as
`<string name="dsp_op">base64...</string>` etc. Get them by any of:

- **Rooted phone / file access:** copy the `shared_prefs` folder here.
- **adb backup (no root):**
  `adb backup -f scio.ab -noapk com.consumerphysics.consumer`
  then point `extract_adb_backup` at `scio.ab`.
- **Loose files:** drop any extracted `dsp_op.bin`, `centers.bin`, ... into
  `01_rawdata/device_files/`.

Set `SOURCE` below to a shared_prefs directory, an `.ab` file, or `None` if you
already have `*.bin` files in `01_rawdata/device_files/`.

In [ ]:
SOURCE = None   # e.g. r"C:\path	o\shared_prefs"  or  r"C:\path	o\scio.ab"
DEVICE_FILES = Path("01_rawdata/device_files")

if SOURCE:
    src = Path(SOURCE)
    scan_dir = src
    if src.suffix == ".ab":
        scan_dir = firmware.extract_adb_backup(src, DEVICE_FILES / "adb_extract")
    found = firmware.scan_shared_prefs_dir(scan_dir)
    if found:
        written = firmware.save_blobs(found, DEVICE_FILES)
        print("extracted:", [p.name for p in written])
    else:
        print("No firmware strings found in", scan_dir)

blobs = firmware.load_blob_dir(DEVICE_FILES)
print("firmware blobs available:", sorted(blobs) or "(none yet)")

## 2. Triage and cipher-signature scan

In [ ]:
if blobs:
    tri = firmware.triage(blobs)
    for name, t in tri.items():
        print(f'{name:18s} {t["size"]:>7d} B  entropy {t["entropy"]:.2f}  LDR={t["valid_ldr"]}  -> {t["verdict"]}')
    if "dsp_op" in blobs:
        sigs = keyrecover.find_signatures(blobs["dsp_op"]["data"])
        print("cipher signatures in dsp_op:",
              {k: ["0x%X" % o for o in v[:4]] for k, v in sigs.items()} or "none")
else:
    print("No firmware yet - section 3 will only try device-identifier keys (low odds).")

## 3. Match firmware checksums to your device (optional)

If you captured file headers in notebook `07`, confirm the firmware you
extracted belongs to *your* SCiO.

In [ ]:
dev_files = sorted(Path("01_rawdata/device_files").glob("device_*.json"))
if dev_files and blobs:
    rec = json.loads(dev_files[-1].read_text())
    headers = {int(k): v.get("checksum") for k, v in rec.get("file_headers", {}).items()}
    for name, (blob_cs, dev_cs, ok) in firmware.match_checksums(blobs, headers).items():
        print(f'{name:18s} blob={blob_cs} device={dev_cs} {"MATCH" if ok else "differ/na"}')
else:
    print("Capture file headers in notebook 07 to enable this check.")

## 4. Load captured scans (and fixtures as a cross-check)

In [ ]:
scans = []
for p in sorted(Path("01_rawdata/scan_json").glob("scan_*.json")):
    try:
        scans.append(store.load_scan(p))
    except Exception as e:
        print("skip", p.name, e)
fixtures = store.load_fixtures()   # have both raw blobs and the server spectrum
print(f"{len(scans)} captured scan(s), {len(fixtures)} fixtures")
device = scans[0]["device"] if scans else (fixtures[0]["device"] if fixtures else {})

## 5. Run key recovery

In [ ]:
use_scans = scans or fixtures
res = keyrecover.recover(use_scans, firmware_blobs=blobs or None, device=device)
print("candidates tried:", res.n_candidates)
if res.best:
    print(f'best oracle score: {res.best["score"]:.3f}  ({res.best["label"]}, '
          f'{res.best["mode"]}/{res.best["iv"]}, view {res.best["view"]})')
print("hits:", len(res.hits))
for h in res.hits[:5]:
    print(f'  {h["label"]:28s} key={h["key"]} {h["mode"]}/{h["iv"]} score={h["score"]:.3f}')
print(); print("=>", res.conclusion)

## 6. If a key was found: decrypt and validate

Decrypt a fixture (which also has the server's `spec_data`) and compare the
derived reflectance to it. This confirms the key and pins down the exact
sample/dark/gradient/white combination. Edit `KEY`, `MODE`, `IV` from the hit
above if needed.

In [ ]:
if res.hits:
    h = res.hits[0]
    KEY, MODE, IV = bytes.fromhex(h["key"]), h["mode"], h["iv"]
    ex = next(e for e in fixtures if "skin" in e["path"])

    def dec(name):
        b = decode.split_blob(ex["blobs"][name])
        plain = decode.decrypt(b.body, KEY, mode=MODE, iv=IV, header=b.header)
        return decode.to_intensity(plain, res.best["view"])

    S, D = dec("sample"), dec("sample_dark")
    W, WD = dec("sample_white"), dec("sample_white_dark")
    R = decode.reflectance(S, D, white=W, white_dark=WD)

    wl = np.arange(740, 740 + 331)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(ex["spectrum"]); ax[0].set_title("server spec_data (reference)")
    ax[1].plot(R); ax[1].set_title("derived from decrypted raw")
    plt.tight_layout(); plt.show()
    print("If these match, the key and pipeline are correct.")
else:
    print("No key yet. Next: disassemble dsp_op (see documentation/firmware_notes.md).")

## 7. If the firmware is encrypted (Lockbox)

If triage reports `dsp_op` as high-entropy with no LDR structure, the BF512's
Lockbox secure-boot has encrypted the code, and neither the code nor the key can
be read from the blob. That is a hard wall for a pure-software approach. The
remaining options (JTAG/OTP readout, a bus tap between the CMOS and the DSP) are
documented in `documentation/firmware_notes.md`. Record whatever you find there
so the next person starts ahead of you.